# 01 · Carbon intensity of the grid

**Goal.** Turn a grid *fuel mix* (how many MWh each fuel produced each hour) into an
hourly **carbon-intensity (CI)** signal in gCO2/kWh -- the price the scheduler later
shops against.

**The math.** For hour $t$, CI is the generation-weighted mean lifecycle emission
factor:

$$ \mathrm{CI}(t) \;=\; \frac{\sum_f \mathrm{gen}_f(t)\,\cdot\,\mathrm{EF}_f}{\sum_f \mathrm{gen}_f(t)} $$

where $\mathrm{gen}_f(t)$ is fuel $f$'s generation (MWh) and $\mathrm{EF}_f$ its
emission factor (gCO2/kWh). The MWh unit cancels top and bottom, so the result is
gCO2 per kWh **consumed**. A wind-heavy hour is clean; an oil/gas-peaker evening is dirty.

The real signal comes from the **EIA Open Data API** (needs `EIA_API_KEY`). With no key
we fall back to a clearly-labelled **DEMO** curve so this notebook runs fully offline.

In [ ]:
# --- standard setup for every notebook in this paper ------------------------
%load_ext autoreload
%autoreload 2
import sys; sys.path.insert(0, "..")                 # find cals (in ../)
from dotenv import load_dotenv; load_dotenv("../.env")  # loads EIA_API_KEY if present
from cals import *          # FACTORS, carbon_intensity, Job, schedule, fifo_baseline, ...
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import nb_utils as U             # guarded data loaders + figure helper (see notebooks/nb_utils.py)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

## 1 · Emission factors

`FACTORS` maps each EIA fuel code to a lifecycle gCO2/kWh value (six are IPCC AR5
medians; `OIL` and `OTH` are documented exceptions). Note `OTH = 230` is a **modeling
choice** (AR5 biomass median used as a proxy for EIA's blended "other" bucket) -- we
sweep its sensitivity in notebook 04.

In [ ]:
factors = pd.Series(FACTORS, name="gCO2/kWh").sort_values()
display(factors.to_frame())
print("cleanest fuel:", factors.idxmin(), factors.min(), "| dirtiest:", factors.idxmax(), factors.max())

## 2 · Fetch the fuel mix and price it (guarded)

`U.get_carbon_intensity()` fetches the real ISO-NE mix when `EIA_API_KEY` is set, else
prints a warning and returns a labelled demo curve. Either way the numbers below flow
through the **same** `carbon_intensity()` math above.

In [ ]:
ci, source = U.get_carbon_intensity()   # (Series indexed by UTC hour, source label)

# `ci` deliberately spans MORE than the reporting year: nb_utils fetches
# CI_START=2019-01-01 .. CI_END=2020-01-02, a +2-day buffer, so an HVAC job in
# the last hours of December still has every hour of its +/-flex window
# priceable and does not silently drop out of the schedule. The SCHEDULER wants
# that buffer; the REPORTED SIGNAL must not include it.
#
# Summary statistics below are therefore restricted to CALENDAR 2019 (8760 h),
# which is the window the paper quotes and the window 01_ci_heatmap.png is drawn
# over. Over the full 8808-hour fetch the median is 260.3; over calendar 2019 it
# is 260.4. Min and max are identical either way (they fall on 2019-06-23 08:00
# and 2019-07-20 16:00 UTC; the 48 buffer hours span only 209.1-295.5), so the
# buffer moves the median alone.
ci_2019 = ci[(ci.index >= "2019-01-01") & (ci.index < "2020-01-01")]

print("source        :", source)
print("hours fetched : %d  (%s .. %s, incl. year-boundary buffer)"
      % (len(ci), ci.index[0].date(), ci.index[-1].date()))
print("hours reported: %d  (calendar 2019 -- the basis for every quoted figure)"
      % len(ci_2019))
print("min / median / max  gCO2/kWh : %.1f / %.1f / %.1f"
      % (ci_2019.min(), ci_2019.median(), ci_2019.max()))
ci.head()

## 3 · See the derivation on a single day

To make the ratio concrete, here is one day of the (demo-shaped) fuel mix and the CI it
produces. Watch CI dip at midday when solar floods in and rise into the evening gas peak.
*(Illustrative fuel mix; with a real key the CI above is the real ISO-NE signal.)*

In [ ]:
day = U.demo_fuel_mix("2018-07-15", "2018-07-16")          # 24 h, long format
wide = day.pivot(index="timestamp", columns="fueltype", values="gen_mwh")
print("generation by fuel (MWh), first hours:"); display(wide.head(3).round(0))

day_ci = carbon_intensity(day)                              # <- the real CI function
fig, (a1, a2) = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
wide.plot.area(ax=a1, linewidth=0, alpha=0.85)
a1.set_ylabel("MWh"); a1.set_title("One day: fuel mix (stacked) and resulting carbon intensity")
a1.legend(ncol=4, fontsize=8, loc="upper left")
a2.plot(day_ci.index, day_ci.values, color="black", lw=2)
a2.set_ylabel("gCO2/kWh"); a2.set_xlabel("hour (UTC)")
fig.tight_layout(); plt.show()

## 4 · Month × hour CI heatmap

Averaging CI over every (month, hour-of-day) cell exposes the two rhythms the scheduler
exploits: a **daily** trough (midday solar / overnight) and a **seasonal** drift. Cheap
(green) cells are where deferrable load wants to move.

In [ ]:
# Convert to ISO-NE STANDARD time (UTC-5) BEFORE pivoting. Relabelling the columns
# alone would leave 60 hours -- the first 5 UTC hours of each month -- filed under the
# wrong month row while the axis displayed standard-time labels, so the row and column
# axes would disagree about what "month" means.
#
# "standard time", NOT "local": ISO-NE observes EDT (UTC-4) for roughly eight months
# of the year. A month x hour climatology needs ONE fixed offset or the columns smear
# across the DST transitions, so we fix UTC-5 and say so on the axis.
#
# The SAME 8,760 hours are plotted as are summarised in the cell above; only their
# (month, hour) attribution changes. Every quoted statistic therefore still holds.
c = ci[(ci.index >= "2019-01-01") & (ci.index < "2020-01-01")]
c_std = c.copy()
c_std.index = c.index - pd.Timedelta(hours=5)
grid = (pd.DataFrame({"ci": c_std.values, "month": c_std.index.month, "hour": c_std.index.hour})
          .groupby(["month", "hour"])["ci"].mean().unstack("hour"))

fig, ax = plt.subplots(figsize=(10, 4.2))
im = ax.imshow(grid.values, aspect="auto", origin="lower", cmap="viridis_r",
               extent=[0, 24, 0.5, 12.5])
ax.set_xticks(range(0, 25, 2)); ax.set_yticks(range(1, 13))
ax.set_yticklabels(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
ax.set_xlabel("hour of day (standard time, UTC$-$5)"); ax.set_ylabel("month")
fig.colorbar(im, ax=ax, label="gCO2/kWh")
ax.grid(False)
U.savefig(fig, "01_ci_heatmap.png"); plt.show()
print("grid min/max cell: %.1f / %.1f" % (grid.values.min(), grid.values.max()))

**Takeaway.** CI is far from flat -- it swings by tens of percent within a day and across
seasons. That variance is exactly the headroom a carbon-aware scheduler converts into
avoided emissions in the next notebooks.